# MCP Evaluation Benchmark — Multi-Server

Generate a synthetic evaluation benchmark for MCP tool-use skills across multiple servers, then validate
that it produces the same model rankings as [Accenture's mcp-bench](https://github.com/Accenture/mcp-bench).

### Pipeline

```
MCP Servers ──> Distillation Flow ──> Synthetic Tasks + Expert Trajectories ──> Model Evaluation ──> Rankings
```

### Servers (6 data-dependent)

| Server | Default Port | Tools | Data type |
|--------|-------------|-------|-----------| 
| Weather Data | 8001 | 4 | Live weather API |
| Medical Calculator | 8002 | 22 | Clinical formulas |
| Wikipedia | 8003 | 9 | Live article content |
| Car Price Evaluator | 8004 | 3 | Vehicle pricing DB |
| Reddit | 8005 | 2 | Live posts/comments |
| DEX Paprika | 8006 | 11 | DeFi/crypto market data |

---
## 0. Setup

### 0.1 Prerequisites

**MCP servers**: Clone [mcp-bench](https://github.com/Accenture/mcp-bench) (provides the server source code) and start the servers:

```bash
git clone https://github.com/Accenture/mcp-bench.git ../mcp-bench
bash start_servers.sh          # installs deps + starts 6 servers on ports 8001-8006
bash start_servers.sh --check  # verify they're running
```

**LangGraph agents**: Each MCP server needs a LangGraph agent connected to it for the task generation step (Section 2).

1. Define a LangGraph graph that connects to your MCP server (e.g., a ReAct agent with MCP tools)
2. Serve it locally: `langgraph dev` (runs on `http://localhost:2024` by default)
3. For multiple servers, deploy one agent per server on different ports
4. Add each agent's URL to `.env`

**Note**: The exploration step may occasionally fail if the frontier model probes edge cases
that cause MCP tool errors (e.g., empty arrays, division by zero). The distillation flow has
built-in quality filtering that handles this — simply re-run the cell if a server fails.
A pre-generated `benchmark_tasks.jsonl` is provided so you can skip Section 2 entirely if needed.

**Environment**: Copy `.env.example` to `.env` and fill in your API key + LangGraph agent URLs.

In [1]:
from pathlib import Path
import asyncio
import json
import os
import sys

from dotenv import load_dotenv
import nest_asyncio
import numpy as np
import pandas as pd

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(NOTEBOOK_DIR / ".env")

# LLM API key used as the default for all OpenAI model calls
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
assert OPENAI_API_KEY and OPENAI_API_KEY != "sk-...", "Set OPENAI_API_KEY in .env"

# Teacher model: used by the distillation flow for question generation + quality scoring (Section 2)
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "openai/gpt-5.2")

# Judge model: LLM-as-judge that scores each evaluated model's trace against the expert gold standard (Section 4)
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "openai/gpt-4o")

# LangGraph API key: optional authentication for LangGraph agent servers (Section 2)
LANGGRAPH_API_KEY = os.environ.get("LANGGRAPH_API_KEY", None)

# ── MCP Server Ports ──────────────────────────────────────────────────
# Configure the port for each MCP server. Default: 8001-8006.
# Each server runs via supergateway on its assigned port.
SERVER_PORTS = {
    "Weather Data": 8001,
    "Medical Calculator": 8002,
    "Wikipedia": 8003,
    "Car Price Evaluator": 8004,
    "Reddit": 8005,
    "DEX Paprika": 8006,
}

# Server -> MCP URL (built from ports)
MCP_SERVERS = {
    name: f"http://localhost:{port}/mcp" for name, port in SERVER_PORTS.items()
}

# Server -> LangGraph agent URL (each agent is connected to one MCP server)
# Deploy one LangGraph agent per server: langgraph dev --port <port>
LANGGRAPH_URLS = {
    "Weather Data": os.environ.get("LANGGRAPH_URL_WEATHER_DATA", ""),
    "Medical Calculator": os.environ.get("LANGGRAPH_URL_MEDICAL_CALCULATOR", ""),
    "Wikipedia": os.environ.get("LANGGRAPH_URL_WIKIPEDIA", ""),
    "Car Price Evaluator": os.environ.get("LANGGRAPH_URL_CAR_PRICE", ""),
    "Reddit": os.environ.get("LANGGRAPH_URL_REDDIT", ""),
    "DEX Paprika": os.environ.get("LANGGRAPH_URL_DEX_PAPRIKA", ""),
}

# ── Models to evaluate ────────────────────────────────────────────────
# Each model can have optional overrides for api_key and api_base.
# Default: uses OPENAI_API_KEY. For local/custom models, set api_base.
#
# Examples:
#   "openai/gpt-4o": {}                                               # OpenAI API
#   "hosted_vllm/my-local-model": {"api_base": "http://localhost:8000/v1"}  # vLLM/sglang
#   "vertex_ai/claude-sonnet-4-6": {"api_key": None}                  # Vertex AI (ADC)
#
MODEL_CONFIGS = {
    "openai/gpt-5": {},
    "openai/gpt-4o": {},
    "openai/gpt-4o-mini": {},
    "hosted_vllm/Qwen3-32B": {"api_base": "http://localhost:30000/v1"},
    "vertex_ai/claude-sonnet-4-6": {"api_key": None},
}

EVAL_MODELS = list(MODEL_CONFIGS.keys())

print(f"Teacher: {TEACHER_MODEL}")
print(f"Judge:   {JUDGE_MODEL}")
print(f"Models:  {EVAL_MODELS}")
print(f"Servers: {list(MCP_SERVERS.keys())}")
print(f"Ports:   {list(SERVER_PORTS.values())}")

Teacher: openai/gpt-5.2
Judge:   openai/gpt-5.2
Models:  ['openai/gpt-5', 'openai/gpt-4o', 'openai/gpt-4o-mini', 'hosted_vllm/Qwen3-32B', 'vertex_ai/claude-sonnet-4-6']
Servers: ['Weather Data', 'Medical Calculator', 'Wikipedia', 'Car Price Evaluator', 'Reddit', 'DEX Paprika']
Ports:   [8001, 8002, 8003, 8004, 8005, 8006]


---
## 1. Discover Tools

Connect to each MCP server and list its tools.

In [2]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def discover_tools(url):
    async with streamablehttp_client(url) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resp = await session.list_tools()
            return [
                {
                    "name": t.name,
                    "description": t.description or "",
                    "inputSchema": t.inputSchema,
                }
                for t in resp.tools
            ]


all_tools = {}
for name, url in MCP_SERVERS.items():
    try:
        tools = asyncio.run(discover_tools(url))
        all_tools[name] = tools
        print(f"  {name}: {len(tools)} tools")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

assert all_tools, "No servers reachable! Start them with: bash start_servers.sh"
print(
    f"\nTotal: {sum(len(t) for t in all_tools.values())} tools across {len(all_tools)} servers"
)

  Weather Data: 4 tools
  Medical Calculator: 22 tools
  Wikipedia: 9 tools
  Car Price Evaluator: 3 tools
  Reddit: 2 tools
  DEX Paprika: 11 tools

Total: 51 tools across 6 servers


---
## 2. Generate Evaluation Tasks

Run the MCP distillation flow on each server at varying complexity levels, then
transform the raw output into a clean evaluation dataset.

The `num_samples` parameter controls how many tools each generated question is designed around:
- `num_samples=2` → simpler questions using 2 tools
- `num_samples=4` → moderate questions using 4 tools
- `num_samples=8` → complex questions using 8 tools

Servers with fewer tools than `num_samples` are automatically skipped for that level.
All tasks are saved to a single `benchmark_tasks.jsonl` file with per-server caching —
servers that already have tasks in the file are skipped on re-run.

In [3]:
from sdg_hub import Flow, FlowRegistry

FlowRegistry.discover_flows()

# ── Configuration ─────────────────────────────────────────────────────
# Tool complexity levels to generate tasks at.
# Each level samples N tools per question from the server's tool set.
# Servers with fewer tools than N are skipped for that level.
NUM_SAMPLES_LEVELS = [2, 4, 8]

# Server descriptions (used in the generation prompt)
SERVER_DESCRIPTIONS = {
    "Weather Data": "Weather data server providing current and forecast weather information.",
    "Medical Calculator": "Medical calculator server with 22 clinical calculation tools.",
    "Wikipedia": "Wikipedia server providing article search, content, and summarization.",
    "Car Price Evaluator": "Vehicle market pricing server for Brazilian car brands.",
    "Reddit": "Reddit server for fetching hot threads and post content.",
    "DEX Paprika": "DeFi/crypto analytics server with pool, token, and network data.",
}


def clean_tool_trace(raw_trace):
    """Normalize Langflow/LangGraph trace to canonical format.

    Canonical format: list of {"name": ..., "input": ..., "output": ...} dicts.
    Strips UI metadata (duration, header, icon) and text-type entries.
    """
    if isinstance(raw_trace, str):
        raw_trace = json.loads(raw_trace)

    cleaned = []
    for entry in raw_trace:
        if not isinstance(entry, dict):
            continue

        if entry.get("type") == "tool_use":
            # LangGraph format: {"type": "tool_use", "tool_calls": [{"name": ..., "args": ...}]}
            if "tool_calls" in entry:
                for tc in entry["tool_calls"]:
                    cleaned.append(
                        {
                            "name": tc.get("name", ""),
                            "input": tc.get("args", {}),
                        }
                    )
            # Langflow format: {"type": "tool_use", "name": ..., "tool_input": ..., "output": ...}
            elif "name" in entry:
                step = {"name": entry["name"], "input": entry.get("tool_input", {})}
                if entry.get("output"):
                    step["output"] = entry["output"]
                cleaned.append(step)

        elif entry.get("type") == "tool_result":
            # LangGraph tool result — attach to previous tool call
            if cleaned and "output" not in cleaned[-1]:
                cleaned[-1]["output"] = entry.get("content", "")

    return cleaned


def extract_expert_tools(trace):
    """Extract ordered list of tool names from a cleaned trace."""
    return [step["name"] for step in trace if "name" in step]


def generate_tasks_for_server(server_name, tools, agent_url, num_samples=2):
    """Run the distillation flow for one server at a given num_samples level."""
    flow_instance = Flow.from_yaml(
        FlowRegistry.get_flow_path("MCP Server Distillation")
    )
    flow_instance.set_model_config(model=TEACHER_MODEL, api_key=OPENAI_API_KEY)

    # Configure AgentBlocks (explore_server + run_expert_trajectory)
    agent_kwargs = {"agent_framework": "langgraph", "agent_url": agent_url}
    if LANGGRAPH_API_KEY:
        agent_kwargs["agent_api_key"] = LANGGRAPH_API_KEY
    flow_instance.set_agent_config(**agent_kwargs)
    flow_instance.set_agent_config(timeout=300, blocks=["explore_server"])

    # Configure AgentResponseExtractorBlocks to use langgraph extraction
    flow_instance.set_agent_config(
        agent_framework="langgraph",
        blocks=["extract_exploration", "extract_agent_text"],
    )

    df = pd.DataFrame(
        {
            "tool_list": [tools],
            "mcp_server_name": [server_name],
            "mcp_server_description": [
                SERVER_DESCRIPTIONS.get(server_name, f"{server_name} MCP server")
            ],
        }
    )

    runtime_params = {}
    if num_samples != 2:
        runtime_params["sample_tools"] = {"num_samples": num_samples}

    result = flow_instance.generate(df, runtime_params=runtime_params)
    result_df = result.to_pandas() if hasattr(result, "to_pandas") else result

    # Keep only the columns needed for transformation
    export_cols = [
        c
        for c in [
            "question",
            "extract_agent_text_text",
            "extract_agent_text_tool_trace",
            "question_quality_rating",
            "completeness_rating",
        ]
        if c in result_df.columns
    ]
    return result_df[export_cols]


def transform_tasks(raw_df, server_name):
    """Transform raw distillation output into clean evaluation format."""
    df = raw_df.copy()
    df["expert_tool_trace"] = df["extract_agent_text_tool_trace"].apply(
        clean_tool_trace
    )
    df["expert_tools"] = df["expert_tool_trace"].apply(extract_expert_tools)
    df = df.rename(columns={"extract_agent_text_text": "expert_answer"})
    df["server"] = server_name
    return df[
        [
            "server",
            "question",
            "expert_answer",
            "expert_tools",
            "expert_tool_trace",
            "question_quality_rating",
            "completeness_rating",
        ]
    ]


print("Flow loaded: MCP Server Distillation")
print(f"Complexity levels: {NUM_SAMPLES_LEVELS}")
print(f"Servers: {list(all_tools.keys())}")

/workspace/home/lab/esivaram/sdg_hub_dev/sdg_hub/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[12:37:01] INFO     Discovered 16 flows                                                             ]8;id=275820;file:///workspace/home/lab/esivaram/sdg_hub_dev/sdg_hub/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=586197;file:///workspace/home/lab/esivaram/sdg_hub_dev/sdg_hub/src/sdg_hub/core/flow/registry.py#126\126]8;;\

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                  ┃ Name                 ┃ Author               ┃ Tags                 ┃ Description          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397    │ Advanced Japanese    │ SDG Hub Contributors │ question-generation, │ A comprehensive flow │
│                     │ Document Grounded    │                      │ knowledge-extractio… │ that generates       │
│                     │ Question-Answer      │                      │ qa-pairs,            │ high-quality         │
│                     │ Generation Flow for  │                      │ document-processing, │ question-answer      │
│                     │ Knowledge Tuning     │                      │ educational,         │ pairs from Japanese  │
│                     │                      │                      │ multilingual,        │ input documents      │
│                     │                      │                      │ japanese             │ using multiple LLM   │
│                     │                      │                      │                      │ blocks for question  │
│                     │                      │                      │                      │ generation, answer   │
│                     │                      │                      │                      │ synthesis, and       │
│                     │                      │                      │                      │ quality evaluation.  │
│ eager-path-837      │ MCP Model Evaluation │ SDG Hub Contributors │ evaluation, mcp,     │ Evaluates an LLM     │
│                     │                      │                      │ benchmark,           │ agent's performance  │
│                     │                      │                      │ model-evaluation,    │ on MCP benchmark     │
│                     │                      │                      │ llm-as-judge         │ tasks. Runs the      │
│                     │                      │                      │                      │ agent against MCP    │
│                     │                      │                      │                      │ servers using fuzzy  │
│                     │                      │                      │                      │ task descriptions,   │
│                     │                      │                      │                      │ then scores the      │
│                     │                      │                      │                      │ resulting            │
│                     │                      │                      │                      │ trajectories with an │
│                     │                      │                      │                      │ LLM-as-judge on task │
│                     │                      │                      │                      │ completion, tool     │
│                     │                      │                      │                      │ usage, and planning  │
│                     │                      │                      │                      │ effectiveness.       │
│                     │                      │                      │                      │ Designed to be run   │
│                     │                      │                      │                      │ once per model being │
│                     │                      │                      │                      │ evaluated.           │
│ epic-jade-656       │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document. Each │
│                     │ Flow                 │          

Flow loaded: MCP Server Distillation
Complexity levels: [2, 4, 8]
Servers: ['Weather Data', 'Medical Calculator', 'Wikipedia', 'Car Price Evaluator', 'Reddit', 'DEX Paprika']


In [4]:
BENCHMARK_PATH = NOTEBOOK_DIR / "benchmark_tasks.jsonl"

# Load existing benchmark (if any) for per-server caching
if BENCHMARK_PATH.exists():
    benchmark_df = pd.read_json(BENCHMARK_PATH, orient="records", lines=True)
    cached_servers = set(benchmark_df["server"].unique())
else:
    benchmark_df = pd.DataFrame()
    cached_servers = set()

new_tasks = []

for server_name, tools in all_tools.items():
    if server_name in cached_servers:
        n = len(benchmark_df[benchmark_df["server"] == server_name])
        print(f"{server_name}: {n} tasks (cached)")
        continue

    agent_url = LANGGRAPH_URLS.get(server_name, "")
    if not agent_url:
        print(f"{server_name}: SKIP — no LangGraph URL in .env")
        continue

    server_tasks = []
    n_tools = len(tools)

    for ns in NUM_SAMPLES_LEVELS:
        if n_tools < ns:
            print(f"  {server_name} ns={ns}: skipped ({n_tools} tools < {ns})")
            continue

        print(f"\n{'─' * 50}")
        print(f"{server_name} — num_samples={ns} ({n_tools} tools)")
        print(f"{'─' * 50}")

        try:
            result_df = generate_tasks_for_server(
                server_name, tools, agent_url, num_samples=ns
            )
            server_tasks.append(result_df)
            print(f"  Generated {len(result_df)} tasks")
        except Exception as e:
            print(f"  FAILED: {e}")

    if server_tasks:
        raw_combined = pd.concat(server_tasks, ignore_index=True)
        transformed = transform_tasks(raw_combined, server_name)
        new_tasks.append(transformed)
        print(f"\n  {server_name}: {len(transformed)} tasks")

if new_tasks:
    new_df = pd.concat(new_tasks, ignore_index=True)
    benchmark_df = pd.concat([benchmark_df, new_df], ignore_index=True)
    benchmark_df.to_json(BENCHMARK_PATH, orient="records", lines=True)
    print(f"\nAdded {len(new_df)} new tasks")

# Summary
print(
    f"\nBenchmark: {len(benchmark_df)} tasks across {benchmark_df['server'].nunique()} servers"
)
print(f"Columns: {list(benchmark_df.columns)}")
print("\nPer server:")
for server, group in benchmark_df.groupby("server"):
    print(f"  {server:<25} {len(group)} tasks")

Weather Data: 17 tasks (cached)
Medical Calculator: 30 tasks (cached)
Wikipedia: 28 tasks (cached)
Car Price Evaluator: 4 tasks (cached)
Reddit: 9 tasks (cached)
DEX Paprika: 23 tasks (cached)

Benchmark: 111 tasks across 6 servers
Columns: ['server', 'question', 'expert_answer', 'expert_tools', 'expert_tool_trace', 'question_quality_rating', 'completeness_rating']

Per server:
  Car Price Evaluator       4 tasks
  DEX Paprika               23 tasks
  Medical Calculator        30 tasks
  Reddit                    9 tasks
  Weather Data              17 tasks
  Wikipedia                 28 tasks


---
## 3. Inspect Tasks

Sample task from the evaluation dataset.

In [5]:
sample = benchmark_df.iloc[0]

print(f"Server: {sample['server']}")
print(f"\nQuestion:\n  {sample['question'][:400]}")
print(f"\nExpert Tools:\n  {sample['expert_tools']}")
print(f"\nQuality: {sample['question_quality_rating']}")
print(f"Completeness: {sample['completeness_rating']}")

print("\nExpert Trajectory:")
for i, step in enumerate(sample["expert_tool_trace"]):
    print(f"  [{i + 1}] {step['name']}({json.dumps(step.get('input', {}))[:80]})")

print(f"\nExpert Answer (first 300 chars):\n  {sample['expert_answer'][:300]}")

Server: Weather Data

Question:
  I’m planning a day trip in Paris, but I mean Paris in Ile-de-France, France (not Paris, Texas). Can you first confirm the right Paris from the search results, then pull the forecast for the next 3 days and tell me which day has the lowest chance of rain (include the date and the max/min temps in °C for that day)? Also, check the current conditions for that same Paris right now (temperature, “feels

Expert Tools:
  ['search_locations_tool', 'get_weather_forecast_tool', 'get_current_weather_tool']

Quality: excellent
Completeness: fully complete

Expert Trajectory:
  [1] search_locations_tool({"query": "Paris"})
  [2] get_weather_forecast_tool({"city": "Paris, Ile-de-France, France", "days": 3})
  [3] get_current_weather_tool({"city": "Paris, Ile-de-France, France"})

Expert Answer (first 300 chars):
  ### Confirming the correct Paris
From the search results, the correct one is:
- **Paris, Ile-de-France, France** (lat **48.87**, lon **2.33**) — *not* Par

---
## 4. Evaluate Models

For each model, run it on all benchmark tasks via `MCPAgentBlock` (direct MCP
connection — no LangGraph needed), then score against the expert gold standard
using the registered **MCP Model Evaluation** flow.

The evaluation pipeline per model:
1. `MCPAgentBlock` runs the model on each task → produces tool traces
2. Traces are extracted and formatted for the judge
3. `MCP Model Evaluation` flow scores traces via LLM-as-judge (6 dimensions)
4. Programmatic metrics (tool recall, precision, order, parameter match) are computed

In [ ]:
from datasets import Dataset
from eval_utils import (
    ZERO_JUDGE,
    ZERO_RESULT,
    compute_tool_metrics,
    extract_model_answer,
    extract_model_tool_trace,
    extract_model_tools,
    format_trace_for_judge,
)
from pydantic import SecretStr

from sdg_hub.core.blocks import MCPAgentBlock

# Load the MCP Model Evaluation flow (judge scoring pipeline)
eval_flow = Flow.from_yaml(FlowRegistry.get_flow_path("MCP Model Evaluation"))
eval_flow.set_model_config(model=JUDGE_MODEL, api_key=OPENAI_API_KEY)

print("Evaluation pipeline loaded:")
print("  Model runner:  MCPAgentBlock (direct MCP)")
print(f"  Judge flow:    MCP Model Evaluation ({len(eval_flow.blocks)} blocks)")
print(f"  Judge model:   {JUDGE_MODEL}")
print("  Dimensions:    task_fulfillment, grounding, tool_appropriateness,")
print(
    "                 parameter_accuracy, dependency_awareness, parallelism_and_efficiency"
)

In [ ]:
# Run evaluation: for each server x each model
# Cached per server x model — only missing combos are evaluated.
# Delete evaluation_results.jsonl to force full re-evaluation.

RESULTS_PATH = NOTEBOOK_DIR / "evaluation_results.jsonl"

if RESULTS_PATH.exists():
    cached_df = pd.read_json(RESULTS_PATH, orient="records", lines=True)
    all_results = cached_df.to_dict("records")
    cached_combos = set(zip(cached_df["server"], cached_df["model"]))
    print(f"Loaded {len(all_results)} cached results")
else:
    all_results = []
    cached_combos = set()

combos_to_run = [
    (s, m)
    for m in EVAL_MODELS
    for s in benchmark_df["server"].unique()
    if (s, m) not in cached_combos
]

if not combos_to_run:
    print(
        f"All {len(EVAL_MODELS)} models x {benchmark_df['server'].nunique()} servers cached."
    )
else:
    print(f"Running {len(combos_to_run)} missing combos")
    task_failures = 0

    for server_name, server_tasks in benchmark_df.groupby("server"):
        server_models = [m for s, m in combos_to_run if s == server_name]
        if not server_models:
            continue

        server_url = MCP_SERVERS.get(server_name)
        if not server_url:
            print(f"SKIP {server_name}: no MCP URL configured")
            continue

        safe_name = server_name.replace(" ", "_")
        print(f"\n{'=' * 60}")
        print(f"{server_name} ({len(server_tasks)} tasks)")
        print(f"{'=' * 60}")

        for model in server_models:
            model_short = model.split("/")[-1]
            config = MODEL_CONFIGS.get(model, {})
            model_api_key = config.get("api_key", OPENAI_API_KEY)
            model_api_base = config.get("api_base", None)

            print(f"\n  {model_short}:", end=" ", flush=True)

            block_kwargs = {
                "block_name": f"eval_{model_short}_{safe_name}",
                "mcp_server_url": server_url,
                "model": model,
                "max_iterations": 20,
                "input_cols": ["question"],
                "output_cols": ["model_trace"],
            }
            if model_api_key is not None:
                block_kwargs["api_key"] = SecretStr(model_api_key)
            if model_api_base is not None:
                block_kwargs["api_base"] = model_api_base

            block = MCPAgentBlock(**block_kwargs)

            # Step 1: Run model — batch first, per-task fallback on failure
            try:
                result_df = block.generate(server_tasks[["question"]].copy())
                traces = result_df["model_trace"].tolist()
            except Exception as e:
                print(f"batch failed ({e}), retrying per-task...", end=" ", flush=True)
                traces = []
                for idx in range(len(server_tasks)):
                    try:
                        single = block.generate(
                            server_tasks.iloc[[idx]][["question"]].copy()
                        )
                        traces.append(single["model_trace"].iloc[0])
                    except Exception:
                        traces.append(None)
                        task_failures += 1

            # Step 2: Extract traces, compute programmatic metrics, prepare judge input
            judge_rows = []
            task_meta = []

            for idx in range(len(server_tasks)):
                task = server_tasks.iloc[idx]
                trace = traces[idx] if idx < len(traces) else None

                if trace is None:
                    all_results.append(
                        {
                            "server": server_name,
                            "model": model,
                            "task_idx": idx,
                            **ZERO_RESULT,
                        }
                    )
                    continue

                try:
                    m_tools = extract_model_tools(trace)
                    m_trace = extract_model_tool_trace(trace)
                    m_answer = extract_model_answer(trace)
                except Exception as e:
                    print(f"\n    Trace extraction failed task {idx}: {e}")
                    all_results.append(
                        {
                            "server": server_name,
                            "model": model,
                            "task_idx": idx,
                            **ZERO_RESULT,
                        }
                    )
                    task_failures += 1
                    continue

                e_tools = task["expert_tools"]
                e_trace = task["expert_tool_trace"]
                metrics = compute_tool_metrics(m_tools, e_tools, m_trace, e_trace)

                # Prepare row for the judge flow
                judge_rows.append(
                    {
                        "question": task["question"],
                        "expert_answer_truncated": task["expert_answer"][:2000],
                        "expert_trace_formatted": format_trace_for_judge(e_trace),
                        "model_answer": m_answer[:2000],
                        "model_trace_formatted": format_trace_for_judge(m_trace),
                    }
                )
                task_meta.append(
                    {
                        "server": server_name,
                        "model": model,
                        "task_idx": idx,
                        **metrics,
                    }
                )

            # Step 3: Run judge flow on all tasks for this model x server
            if judge_rows:
                try:
                    judge_ds = Dataset.from_list(judge_rows)
                    judge_result = eval_flow.generate(judge_ds)
                    judge_df = (
                        judge_result.to_pandas()
                        if hasattr(judge_result, "to_pandas")
                        else pd.DataFrame(judge_result)
                    )

                    judge_cols = [
                        "task_fulfillment",
                        "grounding",
                        "tool_appropriateness",
                        "parameter_accuracy",
                        "dependency_awareness",
                        "parallelism_and_efficiency",
                    ]

                    for i, meta in enumerate(task_meta):
                        scores = {}
                        for col in judge_cols:
                            val = (
                                judge_df[col].iloc[i] if col in judge_df.columns else 0
                            )
                            try:
                                scores[col] = int(val)
                            except (ValueError, TypeError):
                                scores[col] = 0
                        all_results.append({**meta, **scores})

                except Exception as e:
                    print(f"\n    Judge flow failed: {e}")
                    task_failures += len(task_meta)
                    for meta in task_meta:
                        all_results.append({**meta, **ZERO_JUDGE})

            n = sum(
                1
                for r in all_results
                if r["model"] == model and r["server"] == server_name
            )
            avg = np.mean(
                [
                    r.get("task_fulfillment", 0)
                    for r in all_results
                    if r["model"] == model and r["server"] == server_name
                ]
            )
            print(f"{n} tasks, avg_fulfillment={avg:.1f}")

    if task_failures:
        print(f"\nWARNING: {task_failures} task(s) failed — scored 0.")

    pd.DataFrame(all_results).to_json(RESULTS_PATH, orient="records", lines=True)
    print(f"\nSaved {len(all_results)} results to {RESULTS_PATH.name}")

print(f"\nTotal: {len(all_results)} evaluation scores")

---
## 5. Results

In [ ]:
results_df = pd.DataFrame(all_results)

# 6 judge subdimensions (1-10 scale) + 4 programmatic metrics (0-1 scale)
judge_cols = [
    "task_fulfillment",
    "grounding",
    "tool_appropriateness",
    "parameter_accuracy",
    "dependency_awareness",
    "parallelism_and_efficiency",
]
tool_cols = ["tool_recall", "tool_precision", "order_match", "param_match"]
all_cols = tool_cols + judge_cols

# Per-server averaging: mean within each server, then mean across servers.
server_model = results_df.groupby(["server", "model"])[all_cols].mean()
for col in judge_cols:
    server_model[col] = server_model[col] / 10.0
server_model["overall"] = server_model.mean(axis=1)

pivot = server_model["overall"].unstack("model")
pivot.columns = [c.split("/")[-1] for c in pivot.columns]
col_order = pivot.mean().sort_values(ascending=False).index.tolist()
pivot = pivot[col_order]

n_models = results_df["model"].nunique()
task_counts = results_df.groupby("server").size() // n_models
pivot.insert(0, "tasks", task_counts)

pivot.loc["OVERALL"] = pivot.mean()
pivot.loc["OVERALL", "tasks"] = pivot.loc[pivot.index != "OVERALL", "tasks"].sum()

server_rows = pivot.drop("OVERALL").sort_values(col_order[0], ascending=False)
pivot = pd.concat([server_rows, pivot.loc[["OVERALL"]]])

print("Overall Score by Server x Model (per-server averages)")
print("=" * 80)
print(pivot.round(3).to_string())

ranking = sorted(col_order, key=lambda m: pivot.loc["OVERALL", m], reverse=True)
print(f"\nRanking: {' > '.join(ranking)}")

# Dimension breakdown — grouped into 3 aggregates like mcp-bench
print("\nDimension breakdown (all models, per-server average):")
dim_summary = results_df.groupby("model")[judge_cols].mean() / 10.0
dim_summary.index = [m.split("/")[-1] for m in dim_summary.index]
dim_summary = dim_summary.loc[[m for m in ranking if m in dim_summary.index]]

# mcp-bench aggregate groups
dim_summary["task_completion"] = dim_summary[["task_fulfillment", "grounding"]].mean(
    axis=1
)
dim_summary["tool_selection"] = dim_summary[
    ["tool_appropriateness", "parameter_accuracy"]
].mean(axis=1)
dim_summary["planning"] = dim_summary[
    ["dependency_awareness", "parallelism_and_efficiency"]
].mean(axis=1)

agg_cols = ["task_completion", "tool_selection", "planning"]
print(dim_summary[agg_cols + judge_cols].round(3).to_string())

# Save
results_df.to_json(RESULTS_PATH, orient="records", lines=True)
print(f"\nSaved {len(results_df)} evaluation scores to {RESULTS_PATH.name}")